In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# データ分割
from sklearn.model_selection import train_test_split

# 線形モデル
from sklearn.ensemble import RandomForestClassifier

# グラフをアウトプット行に出力するためのマジックコマンド
%matplotlib inline

# 精度評価
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [2]:
# 1. データの読み込み（パスは環境に合わせて調整してください）
# ※ここでは直下に train.csv がある前提にしています
train = pd.read_csv('../../data/raw/train.csv')
test = pd.read_csv('../../data/raw/test.csv')
sample = pd.read_csv('../../data/raw/sample_submission.csv')

In [3]:
# 不要な変数(id、CustomerId)の削除
# IDはそれぞれのユーザーなどが一意であることを示すためだけの変数なので予測にはそこまで寄与しないと考えられる。

train = train.drop(['id','CustomerId'], axis=1)
test = test.drop(['id','CustomerId'], axis=1)

In [4]:
# カテゴリ変数のエンコーディング

# trainとtestを一時的に結合
combined = pd.concat([train, test], ignore_index=True)

# 若年安定フラグ（38歳以下：解約率が極めて低く安定している層）
combined['is_young_stable'] = (combined['Age'] <= 38).astype(int)

# 解約上昇フラグ（39歳以上：解約率が10%を超え、右肩上がりに上昇し始める層）
combined['is_active_churn'] = (combined['Age'] >= 39).astype(int)

# 最危険期フラグ（45歳〜60歳：解約率が50%〜80%に達する最大のピーク層）
combined['is_peak_churn'] = ((combined['Age'] >= 45) & (combined['Age'] <= 60)).astype(int)

# 高齢・退職フラグ（65歳以上：年金受給や退職により、解約率が急減する層。過学習を抑える括り）
combined['is_senior_retire'] = (combined['Age'] >= 65).astype(int)

# カテゴリ変数のエンコーディング
combined_encoded = pd.get_dummies(combined, columns=[
    'Surname', 'Geography', 'Gender', 'IsActiveMember', 'HasCrCard', 'NumOfProducts'
])

# エンコードされたデータを再びtrainとtestに分割
train_encoded = combined_encoded.iloc[:len(train)]
test_encoded = combined_encoded.iloc[len(train):len(train) + 10000]

# 結果を確認
print(train_encoded.shape)
print(test_encoded.shape)

(15000, 854)
(10000, 854)


In [5]:
train_encoded.head()

,CreditScore,Age,Tenure,Balance,EstimatedSalary,Exited,is_young_stable,is_active_churn,is_peak_churn,is_senior_retire,...,Gender_Male,IsActiveMember_0.0,IsActiveMember_1.0,HasCrCard_0.0,HasCrCard_1.0,NumOfProducts_1.0,NumOfProducts_2.0,NumOfProducts_3.0,NumOfProducts_4.0,NumOfProducts_6.0
0,732.0,35.0,2.0,125785.23,52367.29,0.0,1,0,0,0,...,True,True,False,False,True,False,True,False,False,False
1,704.0,31.0,1.0,118788.57,96864.56,0.0,1,0,0,0,...,True,True,False,False,True,True,False,False,False,False
2,706.0,35.0,1.0,0.00,147040.25,0.0,1,0,0,0,...,True,True,False,False,True,False,True,False,False,False
3,850.0,34.0,1.0,0.00,141679.71,0.0,1,0,0,0,...,True,False,True,False,True,False,True,False,False,False
4,468.0,36.0,9.0,0.00,53584.04,0.0,1,0,0,0,...,True,True,False,False,True,True,False,False,False,False


In [6]:
train_encoded.to_csv('../../data/processed/train_age_flag_encoded.csv', index=False)
test_encoded.to_csv('../../data/processed/test_age_flag_encoded.csv', index=False)